In [1]:
!pip install -q kaggle

import os
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

from google.colab import files

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [16]:
import os
from google.colab import userdata

KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
KAGGLE_KEY = userdata.get("KAGGLE_KEY")

if KAGGLE_USERNAME is None or KAGGLE_KEY is None:
    raise ValueError(
        "Kaggle secrets not found. Please add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets."
    )

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

print("Kaggle API credentials loaded safely from Colab Secrets.")


Kaggle API credentials loaded safely from Colab Secrets.


In [17]:
KAGGLE_DATASET = "ninadaithal/imagesoasis"

RAW_DATA_DIR = Path("/content/oasis_raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading dataset:", KAGGLE_DATASET)

subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", KAGGLE_DATASET,
        "-p", str(RAW_DATA_DIR),
        "--unzip"
    ],
    check=True
)

print("Dataset downloaded to:", RAW_DATA_DIR)

Dataset downloaded to: /content/oasis_raw


In [18]:
def show_folder_tree(root_dir, max_depth=3):
    root_dir = Path(root_dir)
    print(f"\nFolder tree for: {root_dir}\n")

    for root, dirs, file_list in os.walk(root_dir):
        root_path = Path(root)
        depth = len(root_path.relative_to(root_dir).parts)

        if depth > max_depth:
            continue

        indent = "    " * depth
        print(f"{indent}{root_path.name}/")

        if depth == max_depth:
            continue


show_folder_tree(RAW_DATA_DIR, max_depth=3)



Folder tree for: /content/oasis_raw

oasis_raw/
    Data/
        Very mild Dementia/
        Mild Dementia/
        Non Demented/
        Moderate Dementia/


In [20]:
READY_DATA_DIR = Path("/content/oasis_ready")
VAL_RATIO = 0.2
SEED = 42

VALID_EXTENSIONS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".npy"]

def clean_class_name(name):
    """
    Converts Kaggle class folder names into simple class names.
    """
    name = name.lower().strip()
    name = name.replace(" ", "_")
    name = name.replace("-", "_")
    name = name.replace("__", "_")

    if "non" in name and "dement" in name:
        return "nondemented"
    if "very" in name and "mild" in name:
        return "very_mild_demented"
    if "moderate" in name:
        return "moderate_demented"
    if "mild" in name:
        return "mild_demented"

    return name

def find_class_folders(source_dir):
    """
    Finds folders that directly contain image files.
    """
    source_dir = Path(source_dir)
    class_folders = []

    for folder in source_dir.rglob("*"):
        if folder.is_dir():
            image_files = [
                f for f in folder.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
            ]
            if len(image_files) > 0:
                class_folders.append(folder)

    return class_folders


class_folders = find_class_folders(RAW_DATA_DIR)

print("\nDetected class folders:")
for folder in class_folders:
    print("-", folder, "->", clean_class_name(folder.name))

if len(class_folders) == 0:
    raise ValueError("No image folders were detected. Please check the downloaded dataset structure.")

# Remove old prepared dataset
if READY_DATA_DIR.exists():
    shutil.rmtree(READY_DATA_DIR)

random.seed(SEED)

for class_folder in class_folders:
    class_name = clean_class_name(class_folder.name)

    image_files = [
        f for f in class_folder.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ]

    random.shuffle(image_files)

    val_count = int(len(image_files) * VAL_RATIO)
    val_files = image_files[:val_count]
    train_files = image_files[val_count:]

    train_out_dir = READY_DATA_DIR / "train" / class_name
    val_out_dir = READY_DATA_DIR / "val" / class_name

    train_out_dir.mkdir(parents=True, exist_ok=True)
    val_out_dir.mkdir(parents=True, exist_ok=True)

    for file_path in train_files:
        shutil.copy(file_path, train_out_dir / file_path.name)

    for file_path in val_files:
        shutil.copy(file_path, val_out_dir / file_path.name)

    print(f"{class_name}: train={len(train_files)}, val={len(val_files)}")

print("\nPrepared dataset saved to:", READY_DATA_DIR)
show_folder_tree(READY_DATA_DIR, max_depth=2)


Detected class folders:
- /content/oasis_raw/Data/Very mild Dementia -> very_mild_demented
- /content/oasis_raw/Data/Mild Dementia -> mild_demented
- /content/oasis_raw/Data/Non Demented -> nondemented
- /content/oasis_raw/Data/Moderate Dementia -> moderate_demented
very_mild_demented: train=10980, val=2745
mild_demented: train=4002, val=1000
nondemented: train=53778, val=13444
moderate_demented: train=391, val=97

Prepared dataset saved to: /content/oasis_ready

Folder tree for: /content/oasis_ready

oasis_ready/
    train/
        nondemented/
        mild_demented/
        moderate_demented/
        very_mild_demented/
    val/
        nondemented/
        mild_demented/
        moderate_demented/
        very_mild_demented/


In [21]:
def count_images_per_class(split_dir):
    split_dir = Path(split_dir)
    counts = {}

    for class_folder in sorted(split_dir.iterdir()):
        if class_folder.is_dir():
            files = [
                f for f in class_folder.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
            ]
            counts[class_folder.name] = len(files)

    return counts


print("\nTrain distribution:")
print(count_images_per_class(READY_DATA_DIR / "train"))

print("\nValidation distribution:")
print(count_images_per_class(READY_DATA_DIR / "val"))



Train distribution:
{'mild_demented': 4002, 'moderate_demented': 391, 'nondemented': 53778, 'very_mild_demented': 10980}

Validation distribution:
{'mild_demented': 1000, 'moderate_demented': 97, 'nondemented': 13444, 'very_mild_demented': 2745}


In [42]:
class CFG:
    DATA_DIR = str(READY_DATA_DIR)
    TRAIN_DIR = str(READY_DATA_DIR / "train")
    VAL_DIR = str(READY_DATA_DIR / "val")

    IMG_SIZE = 128
    CHANNELS = 1

    BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-4

    LATENT_DIM = 64

    # beta controls how strong the KL loss is.
    # For reconstruction baseline, a small beta usually gives clearer images.
    BETA = 0.001

    NUM_WORKERS = 2
    SEED = 42

    OUTPUT_DIR = "/content/outputs_vanilla_cvae"
    CHECKPOINT_DIR = "/content/checkpoints_vanilla_cvae"


os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)

In [43]:
class MRIDataset(Dataset):

    valid_extensions = VALID_EXTENSIONS

    def __init__(self, root_dir, class_to_idx=None, img_size=128):
        self.root_dir = Path(root_dir)
        self.img_size = img_size

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset folder not found: {self.root_dir}")

        class_names = sorted([
            d.name for d in self.root_dir.iterdir()
            if d.is_dir()
        ])

        if len(class_names) == 0:
            raise ValueError(f"No class folders found inside {self.root_dir}")

        if class_to_idx is None:
            self.class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {idx: class_name for class_name, idx in self.class_to_idx.items()}

        self.samples = []

        for class_name in class_names:
            if class_name not in self.class_to_idx:
                continue

            class_folder = self.root_dir / class_name
            label = self.class_to_idx[class_name]

            for file_path in class_folder.rglob("*"):
                if file_path.suffix.lower() in self.valid_extensions:
                    self.samples.append((str(file_path), label))

        if len(self.samples) == 0:
            raise ValueError(f"No images found in {self.root_dir}")

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def load_image(self, file_path):
        file_path = Path(file_path)

        if file_path.suffix.lower() == ".npy":
            arr = np.load(file_path)
            arr = np.squeeze(arr)

            # If a 3D volume is found, take the middle slice.
            if arr.ndim == 3:
                middle = arr.shape[0] // 2
                arr = arr[middle]

            arr = arr.astype(np.float32)
            arr_min = arr.min()
            arr_max = arr.max()

            if arr_max > arr_min:
                arr = (arr - arr_min) / (arr_max - arr_min)
            else:
                arr = np.zeros_like(arr)

            arr = (arr * 255).astype(np.uint8)
            image = Image.fromarray(arr).convert("L")

        else:
            image = Image.open(file_path).convert("L")

        return image

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        image = self.load_image(file_path)
        image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label


train_dataset = MRIDataset(CFG.TRAIN_DIR, img_size=CFG.IMG_SIZE)
val_dataset = MRIDataset(
    CFG.VAL_DIR,
    class_to_idx=train_dataset.class_to_idx,
    img_size=CFG.IMG_SIZE
)

NUM_CLASSES = len(train_dataset.class_to_idx)

print("\nClass mapping:")
print(train_dataset.class_to_idx)
print("Number of classes:", NUM_CLASSES)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)


Class mapping:
{'mild_demented': 0, 'moderate_demented': 1, 'nondemented': 2, 'very_mild_demented': 3}
Number of classes: 4
Train samples: 69151
Validation samples: 17286


In [44]:
images, labels = next(iter(train_loader))
print("\nOne batch image shape:", images.shape)
print("One batch label shape:", labels.shape)
print("Example labels:", labels[:10])

sample_grid_path = os.path.join(CFG.OUTPUT_DIR, "sample_training_images.png")
save_image(images[:16], sample_grid_path, nrow=8)
print("Saved sample image grid:", sample_grid_path)


One batch image shape: torch.Size([32, 1, 128, 128])
One batch label shape: torch.Size([32])
Example labels: tensor([2, 2, 2, 2, 2, 2, 3, 3, 0, 2])
Saved sample image grid: /content/outputs_vanilla_cvae/sample_training_images.png


In [45]:
def label_to_onehot(labels, num_classes):
    return F.one_hot(labels, num_classes=num_classes).float()


def make_label_map(labels, num_classes, height, width):
    """
    Converts labels into spatial label maps.

    Image shape:     [B, 1, H, W]
    Label map shape: [B, C, H, W]
    Combined input:  [B, 1 + C, H, W]
    """
    onehot = label_to_onehot(labels, num_classes)
    label_map = onehot[:, :, None, None]
    label_map = label_map.repeat(1, 1, height, width)
    return label_map

In [46]:
class VanillaCVAE(nn.Module):

    def __init__(self, img_size=128, channels=1, num_classes=4, latent_dim=64):
        super().__init__()
        self.img_size = img_size
        self.channels = channels
        self.num_classes = num_classes
        self.latent_dim = latent_dim

        encoder_input_channels = channels + num_classes

        self.encoder = nn.Sequential(
            nn.Conv2d(encoder_input_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.feature_size = img_size // 16
        self.flatten_dim = 256 * self.feature_size * self.feature_size

        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)

        decoder_input_dim = latent_dim + num_classes
        self.decoder_input = nn.Linear(decoder_input_dim, self.flatten_dim)

        # Correct channel flow:
        # 256 -> 128 -> 64 -> 32 -> 1
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )


    def encode(self, x, labels):
        batch_size, _, height, width = x.shape
        label_map = make_label_map(labels, self.num_classes, height, width).to(x.device)

        encoder_input = torch.cat([x, label_map], dim=1)
        features = self.encoder(encoder_input)
        features = features.view(batch_size, -1)

        mu = self.fc_mu(features)
        logvar = self.fc_logvar(features)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z, labels):
        onehot = label_to_onehot(labels, self.num_classes).to(z.device)
        decoder_input = torch.cat([z, onehot], dim=1)

        x = self.decoder_input(decoder_input)
        x = x.view(-1, 256, self.feature_size, self.feature_size)

        recon = self.decoder(x)
        return recon

    def forward(self, x, labels):
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, labels)
        return recon, mu, logvar

model = VanillaCVAE(
    img_size=CFG.IMG_SIZE,
    channels=CFG.CHANNELS,
    num_classes=NUM_CLASSES,
    latent_dim=CFG.LATENT_DIM
).to(DEVICE)

print("\nModel created successfully.")
print(model)


Model created successfully.
VanillaCVAE(
  (encoder): Sequential(
    (0): Conv2d(5, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2, inplace=True)
    (6): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): LeakyReLU(negative_slope=0.2, inplace=True)
    (9): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (fc_mu): Linear(in_features=16384, out_features=64, b

In [47]:
def cvae_loss(recon, x, mu, logvar, beta=0.001):
    recon_loss = F.mse_loss(recon, x, reduction="mean")
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss


optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE)

In [48]:
def train_one_epoch(model, dataloader, optimizer):
    model.train()

    total_loss_sum = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        recon, mu, logvar = model(images, labels)
        loss, recon_loss, kl_loss = cvae_loss(
            recon, images, mu, logvar, beta=CFG.BETA
        )

        loss.backward()
        optimizer.step()

        total_loss_sum += loss.item()
        recon_loss_sum += recon_loss.item()
        kl_loss_sum += kl_loss.item()

    n = len(dataloader)
    return total_loss_sum / n, recon_loss_sum / n, kl_loss_sum / n

@torch.no_grad()
def validate_one_epoch(model, dataloader):
    model.eval()

    total_loss_sum = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        recon, mu, logvar = model(images, labels)
        loss, recon_loss, kl_loss = cvae_loss(
            recon, images, mu, logvar, beta=CFG.BETA
        )

        total_loss_sum += loss.item()
        recon_loss_sum += recon_loss.item()
        kl_loss_sum += kl_loss.item()

    n = len(dataloader)
    return total_loss_sum / n, recon_loss_sum / n, kl_loss_sum / n


In [49]:
@torch.no_grad()
def save_reconstruction_examples(model, dataloader, epoch, max_images=8):
    """
    Saves original and reconstructed images.

    Top row: original images
    Bottom row: reconstructed images
    """
    model.eval()

    images, labels = next(iter(dataloader))
    images = images[:max_images].to(DEVICE)
    labels = labels[:max_images].to(DEVICE)

    recon, _, _ = model(images, labels)

    comparison = torch.cat([images.cpu(), recon.cpu()], dim=0)
    save_path = os.path.join(CFG.OUTPUT_DIR, f"reconstruction_epoch_{epoch:03d}.png")
    save_image(comparison, save_path, nrow=max_images)

    print("Saved reconstruction example:", save_path)

In [50]:
@torch.no_grad()
def generate_samples_by_class(model, num_samples_per_class=4, file_name="generated_by_class.png"):
    """
    Generates new MRI-like images from random latent vectors.

    Each row represents one dementia class.
    """
    model.eval()

    all_generated = []

    for class_idx in range(NUM_CLASSES):
        z = torch.randn(num_samples_per_class, CFG.LATENT_DIM).to(DEVICE)
        labels = torch.full(
            size=(num_samples_per_class,),
            fill_value=class_idx,
            dtype=torch.long
        ).to(DEVICE)

        generated = model.decode(z, labels)
        all_generated.append(generated.cpu())

    all_generated = torch.cat(all_generated, dim=0)

    save_path = os.path.join(CFG.OUTPUT_DIR, file_name)
    save_image(all_generated, save_path, nrow=num_samples_per_class)
    print("Saved generated samples:", save_path)

In [51]:
best_val_loss = float("inf")
train_history = []
val_history = []

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    train_loss, train_recon, train_kl = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_recon, val_kl = validate_one_epoch(model, val_loader)

    train_history.append({
        "epoch": epoch,
        "loss": train_loss,
        "recon": train_recon,
        "kl": train_kl,
    })

    val_history.append({
        "epoch": epoch,
        "loss": val_loss,
        "recon": val_recon,
        "kl": val_kl,
    })

    print(
        f"Epoch [{epoch:03d}/{CFG.NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.6f} | Train Recon: {train_recon:.6f} | Train KL: {train_kl:.6f} || "
        f"Val Loss: {val_loss:.6f} | Val Recon: {val_recon:.6f} | Val KL: {val_kl:.6f}"
    )

if epoch % 5 == 0:
        save_reconstruction_examples(model, val_loader, epoch)

if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = os.path.join(CFG.CHECKPOINT_DIR, "best_vanilla_cvae.pth")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "class_to_idx": train_dataset.class_to_idx,
            "idx_to_class": train_dataset.idx_to_class,
            "config": {
                "img_size": CFG.IMG_SIZE,
                "channels": CFG.CHANNELS,
                "num_classes": NUM_CLASSES,
                "latent_dim": CFG.LATENT_DIM,
                "beta": CFG.BETA,
            }
        }, checkpoint_path)

        print("Saved best checkpoint:", checkpoint_path)

# Save final reconstruction and generated samples
save_reconstruction_examples(model, val_loader, CFG.NUM_EPOCHS, max_images=8)
generate_samples_by_class(model, num_samples_per_class=4)

print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Outputs saved to:", CFG.OUTPUT_DIR)
print("Checkpoints saved to:", CFG.CHECKPOINT_DIR)

Epoch [001/10] Train Loss: 0.013102 | Train Recon: 0.011887 | Train KL: 1.215095 || Val Loss: 0.006438 | Val Recon: 0.005167 | Val KL: 1.270430
Epoch [002/10] Train Loss: 0.005980 | Train Recon: 0.004693 | Train KL: 1.287061 || Val Loss: 0.005621 | Val Recon: 0.004326 | Val KL: 1.294600
Epoch [003/10] Train Loss: 0.005474 | Train Recon: 0.004198 | Train KL: 1.276538 || Val Loss: 0.005343 | Val Recon: 0.004116 | Val KL: 1.226985
Epoch [004/10] Train Loss: 0.005211 | Train Recon: 0.003945 | Train KL: 1.266230 || Val Loss: 0.005093 | Val Recon: 0.003813 | Val KL: 1.279856
Epoch [005/10] Train Loss: 0.005038 | Train Recon: 0.003778 | Train KL: 1.260435 || Val Loss: 0.004929 | Val Recon: 0.003685 | Val KL: 1.243752
Epoch [006/10] Train Loss: 0.004906 | Train Recon: 0.003651 | Train KL: 1.254582 || Val Loss: 0.004846 | Val Recon: 0.003604 | Val KL: 1.241682
Epoch [007/10] Train Loss: 0.004801 | Train Recon: 0.003551 | Train KL: 1.249847 || Val Loss: 0.004748 | Val Recon: 0.003503 | Val KL: 1

In [52]:
!zip -r /content/outputs_vanilla_cvae.zip /content/outputs_vanilla_cvae
!zip -r /content/checkpoints_vanilla_cvae.zip /content/checkpoints_vanilla_cvae

print("\nYou can download these files from the Colab file panel:")
print("/content/outputs_vanilla_cvae.zip")
print("/content/checkpoints_vanilla_cvae.zip")

  adding: content/outputs_vanilla_cvae/ (stored 0%)
  adding: content/outputs_vanilla_cvae/reconstruction_epoch_010.png (deflated 1%)
  adding: content/outputs_vanilla_cvae/generated_by_class.png (deflated 0%)
  adding: content/outputs_vanilla_cvae/sample_training_images.png (deflated 1%)
  adding: content/checkpoints_vanilla_cvae/ (stored 0%)
  adding: content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth (deflated 8%)

You can download these files from the Colab file panel:
/content/outputs_vanilla_cvae.zip
/content/checkpoints_vanilla_cvae.zip
